In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/zalando-research/fashionmnist/t10k-labels-idx1-ubyte
/kaggle/input/datasets/zalando-research/fashionmnist/t10k-images-idx3-ubyte
/kaggle/input/datasets/zalando-research/fashionmnist/fashion-mnist_test.csv
/kaggle/input/datasets/zalando-research/fashionmnist/fashion-mnist_train.csv
/kaggle/input/datasets/zalando-research/fashionmnist/train-labels-idx1-ubyte
/kaggle/input/datasets/zalando-research/fashionmnist/train-images-idx3-ubyte


In [1]:
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
torch.manual_seed(42)

In [3]:
df = pd.read_csv("/kaggle/input/datasets/zalando-research/fashionmnist/fashion-mnist_train.csv")


In [4]:
x = df.iloc[:, 1 : ]. values
y = df.iloc[:, 0].values

In [5]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)

In [6]:
X_train = X_train/255.0
X_test = X_test/255.0

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [8]:
class CustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype = torch.float32).reshape(-1,1,28,28)
        self.labels = torch.tensor(labels, dtype = torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [9]:
train_dataset = CustomDataset(X_train, y_train)
len(train_dataset)

48000

In [10]:
test_dataset = CustomDataset(X_test, y_test)

In [12]:
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size = 32, shuffle = False, pin_memory = True)

In [30]:
class MyNN(nn.Module):
    def __init__(self, input_features):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(input_features, 32, kernel_size = 3, padding = 'same'),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size = 2, stride = 2),
    
            nn.Conv2d(32, 64, kernel_size = 3, padding = 'same'),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size = 2, stride = 2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, 128),
            nn.ReLU(),
            nn.Dropout(p = 0.4),
    
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(p = 0.4),
    
            nn.Linear(64, 10)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [31]:
EPOCHS = 100
LEARNING_RATE = 0.1

In [32]:
model = MyNN(1)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr = LEARNING_RATE)

In [33]:
len(train_loader)

1500

In [34]:
for epoch in range(EPOCHS):
    total_epoch_loss = 0
    for batch_features, batch_labels in train_loader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        outputs = model(batch_features)
        loss = criterion(outputs, batch_labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_epoch_loss = total_epoch_loss + loss.item()

    avg_loss = total_epoch_loss/len(train_loader)
    print(f'Epoch:{epoch+1}, Loss:{avg_loss}')

Epoch:1, Loss:0.6001765724718571
Epoch:2, Loss:0.4121367147813241
Epoch:3, Loss:0.3554847287138303
Epoch:4, Loss:0.317639832260708
Epoch:5, Loss:0.29271663071587684
Epoch:6, Loss:0.27184440148373445
Epoch:7, Loss:0.24572754553457102
Epoch:8, Loss:0.23216998289090893
Epoch:9, Loss:0.2247285255777339
Epoch:10, Loss:0.20890894083554545
Epoch:11, Loss:0.19809597316694758
Epoch:12, Loss:0.18685919697582723
Epoch:13, Loss:0.18173354516054194
Epoch:14, Loss:0.17411186107061805
Epoch:15, Loss:0.16927386331620314
Epoch:16, Loss:0.16020893607552475
Epoch:17, Loss:0.15415136929950676
Epoch:18, Loss:0.14991259741973287
Epoch:19, Loss:0.14138625699499002
Epoch:20, Loss:0.1405653858849158
Epoch:21, Loss:0.138054530035782
Epoch:22, Loss:0.13139499643992167
Epoch:23, Loss:0.1299041028424787
Epoch:24, Loss:0.12483019923667113
Epoch:25, Loss:0.11695139973059607
Epoch:26, Loss:0.1147779006708879
Epoch:27, Loss:0.11185000464937184
Epoch:28, Loss:0.10900974336545914
Epoch:29, Loss:0.10876978335415091
Epoch

In [35]:
torch.save(model.state_dict(), '/kaggle/working/my_trained_model.pth')


In [36]:
model.load_state_dict(torch.load('/kaggle/working/my_trained_model.pth'))
model.eval()
total = 0
correct = 0
y_pred = []
with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        outputs = model(batch_features)
        _, predicted = torch.max(outputs, 1)
        y_pred.append(predicted)
        total = total + batch_labels.shape[0]
        correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.9213333333333333
